In [ ]:
import pandas as pd
import numpy as np

In [ ]:
file_path = "current.csv"  # coloque current.csv na mesma pasta deste notebook
raw = pd.read_csv(file_path, header=None, low_memory=False)
print(f"Linhas brutas: {len(raw)}  |  Colunas: {raw.shape[1]}")
raw.iloc[:3, :5]

In [ ]:
# Linha 0: cabeçalho com nomes; Linha 1: 'Transform:' + tcodes; Linha 2+: dados
series_names = raw.iloc[0, 1:].tolist()           # nomes das séries
tcode = raw.iloc[1, 1:].astype(float).values      # array de tcodes

df = raw.iloc[2:].copy()
df.columns = ["date"] + series_names
df["date"] = pd.to_datetime(df["date"], format="%m/%d/%Y")
df = df.set_index("date")
df = df.apply(pd.to_numeric, errors="coerce")

print(f"Período: {df.index[0].date()} → {df.index[-1].date()}")
print(f"Séries: {df.shape[1]}  |  tcode únicos: {sorted(set(tcode.astype(int)))}")
df.head(3)

In [ ]:
def apply_tcode(series: pd.Series, code: int) -> pd.Series:
    """
    Aplica a transformação do FRED-MD para induzir estacionariedade.

    Referência: McCracken & Ng (2016), Appendix A.

    Códigos:
        1 - nível (sem transformação)
        2 - primeira diferença:           Δx_t
        3 - segunda diferença:            Δ²x_t
        4 - log:                          ln(x_t)
        5 - primeira diferença do log:    Δln(x_t)
        6 - segunda diferença do log:     Δ²ln(x_t)
        7 - diferença da taxa de cresc.:  Δ(x_t / x_{t-1} - 1)
    """
    x = series.copy()
    if code == 1:
        return x
    elif code == 2:
        return x.diff()
    elif code == 3:
        return x.diff().diff()
    elif code == 4:
        return np.log(x)
    elif code == 5:
        return np.log(x).diff()
    elif code == 6:
        return np.log(x).diff().diff()
    elif code == 7:
        return x.pct_change().diff()
    else:
        raise ValueError(f"tcode desconhecido: {code}")

In [ ]:
transformed = {}
for col, code in zip(df.columns, tcode):
    transformed[col] = apply_tcode(df[col], int(code))

df_transformed = pd.DataFrame(transformed, index=df.index)
df_transformed.dropna(how="all", inplace=True)

print(f"Shape transformado: {df_transformed.shape}")
df_transformed.head(3)

In [ ]:
_tcode_labels = {
    1: "nível",
    2: "Δx",
    3: "Δ²x",
    4: "ln(x)",
    5: "Δln(x)",
    6: "Δ²ln(x)",
    7: "Δ(pct_change)",
}

summary = pd.DataFrame(
    {
        "tcode": tcode.astype(int),
        "transformacao": [_tcode_labels[int(c)] for c in tcode],
    },
    index=df.columns,
)
summary.index.name = "serie"

print("Distribuição de transformações:")
print(summary["transformacao"].value_counts().to_string())
print()
summary

In [ ]:
output_path = "fred_md_transformed.csv"
df_transformed.to_csv(output_path)
print(f"Arquivo salvo: {output_path}  ({df_transformed.shape[0]} obs × {df_transformed.shape[1]} séries)")

In [ ]:
# Seleciona automaticamente a primeira série sem nenhum NaN
serie_teste = df_transformed.columns[df_transformed.isna().sum() == 0][0]

s = df_transformed[serie_teste]
s_z = (s - s.mean()) / s.std()

print(f"Série escolhida: {serie_teste}")
print(f"Observações: {len(s_z)}")
print(f"Média após z-score:        {s_z.mean():.10f}  (esperado ≈ 0)")
print(f"Desvio padrão após z-score: {s_z.std():.10f}  (esperado ≈ 1)")
s_z.head(10)